<a href="https://colab.research.google.com/github/nimra231/flyrank-ml-internship/blob/main/week02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
!git clone https://github.com/nimra231/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 60 (delta 14), reused 31 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 1.63 MiB | 13.95 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship


In [18]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


**My ML Task: Content Refresh Prioritization**

The goal of my machine learning task is to identify which existing content is losing its search performance and should be updated first. The model predicts a **continuous score** that shows how much a page's performance is declining. A higher score means the content needs more urgent attention, while a lower score means it is still performing well.

This is a **scoring (regression)** task because the model outputs a numerical value instead of assigning content to predefined categories (classification) or grouping similar content without labels (clustering). Although these scores can later be used to prioritize or rank content for updates, the model's main job is to **predict the decline score**, not to directly rank or classify the content.


**Target (Proxy)**

Ideally, the model would predict how much a piece of content's search performance will decline in the future. However, since a static dataset only contains historical data, the actual future decline is not available and cannot be used as the target. Instead, I use **`trend_pct`** as a proxy target. This column compares the content's recent performance (such as clicks and impressions in the last 30 days) with the previous 30 days. A negative `trend_pct` indicates that the content is losing search performance, while a positive value shows improvement. This makes it a reasonable measure of content decline. The dataset also includes **`trend_direction`** (up, down, or stable), which is derived from the same comparison. It is useful as a reference or for checking consistency, but it is not the value the model is predicting. Although `trend_pct` is a practical substitute, it is still a proxy and not the actual future decline, so it cannot perfectly represent long-term content performance.


**Success Metric**

The primary success metric for this task is **Spearman rank correlation** because the model is used to prioritize which content should be updated first. In this use case, it is more important that the model ranks the most rapidly declining pages higher than it is to predict the exact decline percentage. Spearman rank correlation measures how closely the predicted order matches the actual order of content decline, making it well suited for this prioritization task. As a secondary metric, **RMSE (Root Mean Squared Error)** can be used to measure how close the predicted `trend_pct` values are to the actual values. While RMSE helps verify that the predicted scores are on a reasonable scale, it is less important than correctly ranking the content. A classification metric such as **accuracy** is not appropriate because the model predicts a continuous numerical value rather than assigning content to predefined categories.


In [19]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Columns you'd know about a piece of content before predicting its decline
feature_cols = [
    "content_type", "main_intent", "word_count", "char_count",
    "search_volume", "competition", "competition_level", "cpc",
    "content_age_days", "age_tier", "days_since_last_update", "freshness_tier",
    "avg_position", "position_tier", "provider_used", "model_used",
]

lane_df = df[["content_id", "client_id"] + feature_cols + ["trend_pct", "trend_direction"]].copy()

print(lane_df.shape)
lane_df.dtypes

(30000, 20)


,0
content_id,object
client_id,object
content_type,object
main_intent,object
word_count,float64
char_count,float64
search_volume,float64
competition,float64
competition_level,object
cpc,float64


**Review of the ML Framing**

The overall ML framing is consistent. The task type is **scoring (regression)**, and the chosen evaluation metrics—**Spearman rank correlation** as the primary metric and **RMSE** as a secondary metric—are appropriate for predicting a continuous decline score. The target column, **`target_trend_pct`**, is present in the dataset, and the model's output is intended to support content refresh prioritization. Since the goal is to identify which content should be updated first, preserving the correct ordering of content is more important than predicting the exact decline percentage, making Spearman rank correlation the most suitable primary metric.

There are two important limitations to acknowledge. First, **`target_trend_pct`** is only a proxy for future performance decline rather than a direct measure of future outcomes, so the model learns from a reasonable substitute instead of true ground truth. Second, the target contains missing values (approximately 3,388 rows) and extreme outliers (up to 44,900%), both of which would need to be handled through appropriate preprocessing before training a reliable machine learning model.
